# 🧠 Step 9 — Neural Network Training

## Architecture
```
File 1: Numerical features          File 2: FinBERT embeddings
(OHLC log returns + indicators)     (768-dim per date)
         ↓                                   ↓
   Linear Transformer              Already encoded by FinBERT
   (captures temporal                        ↓
    dependencies)                    Projection layer
         ↓                          (768 → 64 dims)
    Numerical vector                         ↓
         └──────────── CONCATENATE ──────────┘
                            ↓
              Feedforward Neural Network
              (GELU · BatchNorm · Dropout)
                            ↓
                 Predicted next-day
                 Nifty 50 log return
                            ↓
               Inverse transform → price
```

## Loss Function
**Huber Loss** — sensitive to small errors, robust to large market anomalies

## Inputs
- `nifty50_step4_standard_train.csv` — normalised numerical features (train)
- `nifty50_step4_standard_test.csv`  — normalised numerical features (test)
- `step7b_finbert_embeddings.csv`    — 768-dim FinBERT embeddings per date
- `standard_scaler.pkl`              — scaler for inverse transform

## Outputs
- `step9_predictions.csv`            — predicted vs actual prices
- `step9_model_weights.pth`          — saved model weights
- `step9_evaluation_metrics.csv`     — MAE, RMSE, MAPE, Directional Accuracy

---
**Enable GPU first:** Runtime → Change runtime type → T4 GPU → Save

Then: **Runtime → Run All**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Install Libraries
# ─────────────────────────────────────────────────────────────────────────
!pip install torch pandas numpy scikit-learn joblib -q
print('✓ Libraries ready')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Imports and Configuration
# ─────────────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import joblib
import os
import math
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Device: {device}')
if str(device) == 'cpu':
    print('  ⚠️  No GPU — enable via Runtime → Change runtime type → T4 GPU')

# ── MODEL CONFIG ──────────────────────────────────────────────────────────
SEQUENCE_LENGTH   = 20      # days of history fed into Linear Transformer
EMBED_DIM         = 64      # transformer embedding dimension
NUM_HEADS         = 4       # attention heads
NUM_LAYERS        = 2       # transformer layers
TEXT_PROJ_DIM     = 64      # FinBERT 768 → 64 projection
HIDDEN_DIM        = 256     # FNN hidden layer size
DROPOUT           = 0.2     # dropout rate
LEARNING_RATE     = 3e-5    # Adam learning rate (lower = more stable)
EPOCHS            = 200     # max training epochs
BATCH_SIZE        = 16      # smaller batch = more gradient updates
PATIENCE          = 25      # early stopping patience
HUBER_DELTA       = 0.5     # Huber loss delta (smaller for log returns)
WARMUP_EPOCHS     = 10      # gradually increase LR for first 10 epochs

print(f'\n  Model config:')
print(f'    Sequence length : {SEQUENCE_LENGTH} days')
print(f'    Transformer     : {NUM_LAYERS} layers, {NUM_HEADS} heads, dim={EMBED_DIM}')
print(f'    FinBERT proj    : 768 → {TEXT_PROJ_DIM} dims')
print(f'    FNN hidden      : {HIDDEN_DIM}')
print(f'    Dropout         : {DROPOUT}')
print(f'    Epochs          : {EPOCHS} (with early stopping patience={PATIENCE})')
print(f'    Loss            : Huber (delta={HUBER_DELTA})')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — Upload and Load All Input Files
# ─────────────────────────────────────────────────────────────────────────

print('Upload 1/5: nifty50_step4_standard_train.csv')
u = files.upload(); train_num_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(train_num_df)} train rows, {len(train_num_df.columns)} columns')

print('\nUpload 2/5: nifty50_step4_standard_test.csv')
u = files.upload(); test_num_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(test_num_df)} test rows, {len(test_num_df.columns)} columns')

print('\nUpload 3/5: step7b_finbert_embeddings.csv')
u = files.upload(); emb_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(emb_df)} embedding rows, {len(emb_df.columns)} columns')

print('\nUpload 4/5: step7a_finbert_sentiment_comparison.csv')
u = files.upload(); sent_df = pd.read_csv(list(u.keys())[0])
print(f'  ✓ {len(sent_df)} sentiment rows')

print('\nUpload 5/5: standard_scaler.pkl')
u = files.upload(); scaler = joblib.load(list(u.keys())[0])
print(f'  ✓ Scaler loaded')

# ── Parse dates ────────────────────────────────────────────────────────────
for df in [train_num_df, test_num_df]:
    df['date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce').dt.strftime('%Y-%m-%d')

emb_df['date'] = pd.to_datetime(
    emb_df['date'], dayfirst=True, errors='coerce'
).dt.strftime('%Y-%m-%d')

sent_df['date'] = pd.to_datetime(
    sent_df['date'], dayfirst=True, errors='coerce'
).dt.strftime('%Y-%m-%d')
sent_df = sent_df.dropna(subset=['date'])
for col in ['finbert_score','gpt4o_score',
            'finbert_pos_prob','finbert_neg_prob','finbert_neu_prob']:
    if col in sent_df.columns:
        sent_df[col] = pd.to_numeric(sent_df[col], errors='coerce').fillna(0)

# ── Feature columns ────────────────────────────────────────────────────────
# LogReturn_Close excluded from features — TARGET only
# Including it would cause data leakage (model copies prev value → low loss)
EXCLUDE_COLS  = ['Date', 'date', 'Close', 'LogReturn_Close']
NUM_FEAT_COLS = [c for c in train_num_df.columns if c not in EXCLUDE_COLS]
TARGET_COL    = 'LogReturn_Close'
EMB_COLS      = [c for c in emb_df.columns if c.startswith('emb_')]
SENT_COLS     = [c for c in ['finbert_score','gpt4o_score',
                              'finbert_pos_prob','finbert_neg_prob','finbert_neu_prob']
                 if c in sent_df.columns]

print(f'\n  Numerical features : {len(NUM_FEAT_COLS)}  {NUM_FEAT_COLS}')
print(f'  Embedding dims     : {len(EMB_COLS)}')
print(f'  Sentiment features : {SENT_COLS}')
print(f'  Target             : {TARGET_COL}')
print(f'  Train dates        : {train_num_df["date"].iloc[0]} → {train_num_df["date"].iloc[-1]}')
print(f'  Test dates         : {test_num_df["date"].iloc[0]} → {test_num_df["date"].iloc[-1]}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — Merge Numerical + Embeddings + Sentiment Features
# ─────────────────────────────────────────────────────────────────────────

def merge_features(num_df, emb_df, sent_df, num_cols, emb_cols, sent_cols, target_col):
    cols_needed = ['date', target_col] + [c for c in num_cols if c != target_col]
    merged = pd.merge(num_df[cols_needed],
                      emb_df[['date'] + emb_cols],
                      on='date', how='left')
    if sent_cols:
        merged = pd.merge(merged,
                          sent_df[['date'] + sent_cols],
                          on='date', how='left')
        merged[sent_cols] = merged[sent_cols].fillna(0)
    merged[emb_cols] = merged[emb_cols].fillna(0)
    merged = merged.dropna(subset=num_cols).reset_index(drop=True)
    return merged

train_merged = merge_features(train_num_df, emb_df, sent_df,
                               NUM_FEAT_COLS, EMB_COLS, SENT_COLS, TARGET_COL)
test_merged  = merge_features(test_num_df,  emb_df, sent_df,
                               NUM_FEAT_COLS, EMB_COLS, SENT_COLS, TARGET_COL)

# All feature columns for model
ALL_FEAT_COLS = NUM_FEAT_COLS + SENT_COLS

# Embedding coverage check
train_emb_cov = (train_merged[EMB_COLS[0]] != 0).mean() * 100
test_emb_cov  = (test_merged[EMB_COLS[0]]  != 0).mean() * 100
sent_cov      = (train_merged[SENT_COLS[0]] != 0).mean() * 100 if SENT_COLS else 0

print(f'  Train merged shape     : {train_merged.shape}')
print(f'  Test merged shape      : {test_merged.shape}')
print(f'  Train emb coverage     : {train_emb_cov:.1f}%  (% of days with embeddings)')
print(f'  Test emb coverage      : {test_emb_cov:.1f}%')
print(f'  Train sentiment cov    : {sent_cov:.1f}%')
print(f'  Target col             : {TARGET_COL}')
print(f'  Target stats           : mean={train_merged[TARGET_COL].mean():.4f}  std={train_merged[TARGET_COL].std():.4f}')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — Build Sequence Dataset
#
# Creates sliding windows of SEQUENCE_LENGTH days.
# Each sample: features[t-20:t] → target[t]
# ─────────────────────────────────────────────────────────────────────────

class StockDataset(Dataset):
    def __init__(self, df, num_cols, emb_cols, sent_cols, target_col, seq_len):
        self.seq_len    = seq_len
        self.num_data   = df[num_cols].values.astype(np.float32)
        # Concatenate embeddings + sentiment into one text vector
        all_txt = emb_cols + sent_cols
        self.emb_data   = df[all_txt].values.astype(np.float32)
        self.targets    = df[target_col].values.astype(np.float32)
        self.n_samples  = len(df) - seq_len

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        # Sequence of numerical features: shape (seq_len, num_features)
        num_seq = self.num_data[idx : idx + self.seq_len]
        # FinBERT embedding for the last day in the sequence
        emb_vec = self.emb_data[idx + self.seq_len - 1]
        # Target: next day's log return
        target  = self.targets[idx + self.seq_len]
        return (
            torch.tensor(num_seq, dtype=torch.float32),
            torch.tensor(emb_vec, dtype=torch.float32),
            torch.tensor(target,  dtype=torch.float32),
        )


train_dataset = StockDataset(
    train_merged, NUM_FEAT_COLS, EMB_COLS, SENT_COLS, TARGET_COL, SEQUENCE_LENGTH
)
test_dataset  = StockDataset(
    test_merged,  NUM_FEAT_COLS, EMB_COLS, SENT_COLS, TARGET_COL, SEQUENCE_LENGTH
)

train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
test_loader   = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

NUM_FEATURES  = len(NUM_FEAT_COLS)
EMB_DIM       = len(EMB_COLS) + len(SENT_COLS)  # 768 + sentiment

print(f'  Train samples : {len(train_dataset)}')
print(f'  Test samples  : {len(test_dataset)}')
print(f'  Num features  : {NUM_FEATURES}')
print(f'  Emb dim       : {EMB_DIM}')
print(f'  Sequence len  : {SEQUENCE_LENGTH} days')

# Verify batch shape
num_batch, emb_batch, tgt_batch = next(iter(train_loader))
print(f'\n  Batch shapes:')
print(f'    Numerical seq : {num_batch.shape}  (batch, seq_len, num_features)')
print(f'    Embedding vec : {emb_batch.shape}  (batch, 768)')
print(f'    Target        : {tgt_batch.shape}  (batch,)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Define Model Architecture
#
# Architecture:
#   Numerical branch:
#     Input projection → Linear Transformer → context vector
#   Textual branch:
#     FinBERT embedding (768) → Linear projection → 64-dim vector
#   Fusion:
#     Concatenate both vectors → FNN → predicted log return
# ─────────────────────────────────────────────────────────────────────────

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for the transformer."""
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class MultimodalStockPredictor(nn.Module):
    def __init__(self, num_features, emb_dim, embed_dim, num_heads,
                 num_layers, text_proj_dim, hidden_dim, dropout):
        super().__init__()

        # ── Numerical branch: Linear Transformer ──────────────────────────
        self.num_proj    = nn.Linear(num_features, embed_dim)
        self.pos_enc     = PositionalEncoding(embed_dim, dropout=dropout)
        encoder_layer    = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout, batch_first=True,
            activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # ── Textual branch: FinBERT projection ───────────────────────────
        self.text_proj   = nn.Sequential(
            nn.Linear(emb_dim, text_proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # ── Fusion FNN ────────────────────────────────────────────────────
        fusion_dim = embed_dim + text_proj_dim
        self.fnn   = nn.Sequential(
            nn.Linear(fusion_dim, hidden_dim),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, num_seq, emb_vec):
        # Numerical branch
        x = self.num_proj(num_seq)      # (B, seq_len, embed_dim)
        x = self.pos_enc(x)
        x = self.transformer(x)         # (B, seq_len, embed_dim)
        x = x.mean(dim=1)              # mean pool → (B, embed_dim)

        # Textual branch
        t = self.text_proj(emb_vec)     # (B, text_proj_dim)

        # Fusion
        fused = torch.cat([x, t], dim=1)  # (B, embed_dim + text_proj_dim)
        out   = self.fnn(fused)            # (B, 1)
        return out.squeeze(-1)             # (B,)


model = MultimodalStockPredictor(
    num_features  = NUM_FEATURES,
    emb_dim       = EMB_DIM,
    embed_dim     = EMBED_DIM,
    num_heads     = NUM_HEADS,
    num_layers    = NUM_LAYERS,
    text_proj_dim = TEXT_PROJ_DIM,
    hidden_dim    = HIDDEN_DIM,
    dropout       = DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Model architecture:')
print(model)
print(f'\n  Total trainable parameters: {total_params:,}')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — Train the Model
#
# Training setup:
#   Loss     : Huber Loss (robust to extreme market events)
#   Optimiser: Adam with cosine annealing LR schedule
#   Stopping : Early stopping on validation loss (patience=15)
# ─────────────────────────────────────────────────────────────────────────

criterion  = nn.HuberLoss(delta=HUBER_DELTA)
optimizer  = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)

# Warmup + cosine annealing schedule
def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return float(epoch + 1) / float(WARMUP_EPOCHS)  # linear warmup
    progress = (epoch - WARMUP_EPOCHS) / (EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1.0 + math.cos(math.pi * progress))  # cosine decay

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

best_val_loss  = float('inf')
patience_count = 0
train_losses   = []
val_losses     = []
best_epoch     = 0

print('=' * 60)
print('  TRAINING')
print('=' * 60)
print(f'  Loss      : Huber (delta={HUBER_DELTA})')
print(f'  Optimiser : Adam (lr={LEARNING_RATE})')
print(f'  Schedule  : Cosine annealing')
print(f'  Patience  : {PATIENCE} epochs')
print()

for epoch in range(1, EPOCHS + 1):

    # ── Training phase ─────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0
    for num_seq, emb_vec, target in train_loader:
        num_seq = num_seq.to(device)
        emb_vec = emb_vec.to(device)
        target  = target.to(device)

        optimizer.zero_grad()
        pred  = model(num_seq, emb_vec)
        loss  = criterion(pred, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # ── Validation phase ───────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for num_seq, emb_vec, target in test_loader:
            num_seq = num_seq.to(device)
            emb_vec = emb_vec.to(device)
            target  = target.to(device)
            pred    = model(num_seq, emb_vec)
            loss    = criterion(pred, target)
            val_loss += loss.item()
    val_loss /= len(test_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step()

    # ── Print every 10 epochs ──────────────────────────────────────────────
    if epoch % 5 == 0 or epoch == 1:
        lr = optimizer.param_groups[0]['lr']
        print(f'  Epoch {epoch:>3}/{EPOCHS}  '
              f'Train={train_loss:.6f}  '
              f'Val={val_loss:.6f}  '
              f'LR={lr:.2e}')

    # ── Early stopping ────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        best_epoch     = epoch
        patience_count = 0
        torch.save(model.state_dict(), 'step9_model_weights.pth')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'\n  Early stopping at epoch {epoch}')
            print(f'  Best epoch  : {best_epoch}')
            print(f'  Best val loss: {best_val_loss:.6f}')
            break

print(f'\n✅ Training complete')
print(f'  Best epoch     : {best_epoch}')
print(f'  Best val loss  : {best_val_loss:.6f}')
print(f'  Model saved    : step9_model_weights.pth')

# Load best weights for evaluation
model.load_state_dict(torch.load('step9_model_weights.pth', map_location=device))
print('  Best weights loaded for evaluation')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 8 — Evaluate Model
#
# Metrics:
#   MAE                — Mean Absolute Error
#   RMSE               — Root Mean Squared Error
#   MAPE               — Mean Absolute Percentage Error
#   Directional Accuracy — % of days where direction (up/down) is correct
#   R²                 — Proportion of variance explained
# ─────────────────────────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model.eval()
all_preds   = []
all_targets = []

with torch.no_grad():
    for num_seq, emb_vec, target in test_loader:
        num_seq = num_seq.to(device)
        emb_vec = emb_vec.to(device)
        pred    = model(num_seq, emb_vec).cpu().numpy()
        all_preds.extend(pred)
        all_targets.extend(target.numpy())

preds   = np.array(all_preds)
actuals = np.array(all_targets)

# ── Metrics on normalised log return scale ────────────────────────────────
mae       = mean_absolute_error(actuals, preds)
rmse      = np.sqrt(mean_squared_error(actuals, preds))
r2        = r2_score(actuals, preds)

# MAPE — for log returns use a threshold to avoid near-zero division
# Log returns near 0 produce artificially huge MAPE — use 1e-4 threshold
nonzero   = np.abs(actuals) > 1e-4
if nonzero.sum() > 0:
    mape  = np.mean(np.abs((actuals[nonzero] - preds[nonzero]) / actuals[nonzero])) * 100
else:
    mape  = np.nan
    print('  ⚠️  MAPE skipped — log returns too close to zero')

# Directional Accuracy
dir_actual = np.sign(actuals)
dir_pred   = np.sign(preds)
dir_acc    = np.mean(dir_actual == dir_pred) * 100

print('=' * 60)
print('  EVALUATION METRICS (on normalised log return scale)')
print('=' * 60)
print(f'  MAE                  : {mae:.6f}')
print(f'  RMSE                 : {rmse:.6f}')
print(f'  MAPE                 : {mape:.2f}%')
print(f'  R²                   : {r2:.4f}')
print(f'  Directional Accuracy : {dir_acc:.2f}%')
print()
print('  Interpretation:')
print(f'    Directional accuracy > 50% means model predicts direction')
print(f'    better than random — yours is {dir_acc:.1f}%')
if dir_acc > 55:
    print(f'    ✅ Good directional signal')
elif dir_acc > 50:
    print(f'    ⚠️  Marginally better than random')
else:
    print(f'    ❌ Below random — try increasing EPOCHS or adjusting LR')
print()
print('  Note: MAPE is unreliable for log returns near zero.')
print('  Focus on MAE, RMSE and Directional Accuracy for log return prediction.')

# Save metrics
metrics_df = pd.DataFrame([{
    'model':                'Multimodal (Linear Transformer + FinBERT + FNN)',
    'MAE':                  round(mae, 6),
    'RMSE':                 round(rmse, 6),
    'MAPE':                 round(mape, 2),
    'R2':                   round(r2, 4),
    'Directional_Accuracy': round(dir_acc, 2),
    'Best_Epoch':           best_epoch,
    'Best_Val_Loss':        round(best_val_loss, 6),
}])
metrics_df.to_csv('step9_evaluation_metrics.csv', index=False)
print(f'\n  Metrics saved → step9_evaluation_metrics.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 9 — Generate Predictions and Save
# ─────────────────────────────────────────────────────────────────────────

# Get test dates
test_dates = test_merged['date'].values[SEQUENCE_LENGTH:]
test_dates = test_dates[:len(preds)]  # align with predictions

# Build predictions DataFrame
pred_df = pd.DataFrame({
    'date':              test_dates,
    'actual_log_return': actuals,
    'pred_log_return':   preds,
    'actual_direction':  np.where(actuals >= 0, 'UP', 'DOWN'),
    'pred_direction':    np.where(preds   >= 0, 'UP', 'DOWN'),
    'direction_correct': dir_actual == dir_pred,
    'error':             np.abs(actuals - preds),
})

pred_df['date'] = pd.to_datetime(pred_df['date']).dt.strftime('%d-%b-%Y')

print('  Sample predictions (first 10 rows):')
print(pred_df[['date','actual_log_return','pred_log_return',
               'actual_direction','pred_direction','direction_correct']]
      .head(10).to_string(index=False))

# Save
pred_df.to_csv('step9_predictions.csv', index=False)
print(f'\n  Predictions saved → step9_predictions.csv')

# Training curve
loss_df = pd.DataFrame({
    'epoch':      list(range(1, len(train_losses)+1)),
    'train_loss': train_losses,
    'val_loss':   val_losses,
})
loss_df.to_csv('step9_training_loss.csv', index=False)
print(f'  Training loss curve saved → step9_training_loss.csv')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CELL 10 — Download All Output Files
# ─────────────────────────────────────────────────────────────────────────
output_files = [
    ('step9_predictions.csv',        'Predicted vs actual log returns per date'),
    ('step9_evaluation_metrics.csv', 'MAE, RMSE, MAPE, R², Directional Accuracy'),
    ('step9_training_loss.csv',      'Train and validation loss per epoch'),
    ('step9_model_weights.pth',      'Saved best model weights'),
]

print('=' * 60)
print('  STEP 9 COMPLETE — ALL OUTPUT FILES')
print('=' * 60)
for fname, desc in output_files:
    size = os.path.getsize(fname)
    print(f'  ✓ {fname}')
    print(f'      {desc}  ({size/1024:.1f} KB)')

print()
print('  Final metrics summary:')
print(f'    MAE                  : {mae:.6f}')
print(f'    RMSE                 : {rmse:.6f}')
print(f'    MAPE                 : {mape:.2f}%')
print(f'    R²                   : {r2:.4f}')
print(f'    Directional Accuracy : {dir_acc:.2f}%')
print(f'    Best epoch           : {best_epoch}')

print()
print('⬇️  Downloading all files...')
for fname, _ in output_files:
    files.download(fname)
    print(f'  ✓ {fname} downloaded')

print()
print('✅ Step 9 Complete — Full pipeline done!')
print()
print('  Your two-file input to the neural network:')
print('  File 1 → Normalised numerical features (Step 4)')
print('  File 2 → FinBERT 768-dim embeddings    (Step 7)')
print()
print('  Use step9_evaluation_metrics.csv in your results chapter')
print('  Use step9_predictions.csv for prediction vs actual charts')
print('  Use step9_training_loss.csv for loss curve visualisation')